# Caregiver AI Lab — Colab Workflow

End-to-end experimentation for Egyptian-Arabic care-note summarization.

⚠️ **Not a medical tool.** Synthetic fictional data only. Never diagnose or recommend treatment with this.

In [ ]:

import os, sys
# Make the repo importable whether opened from GitHub or Drive
for p in ['/content/caregiver-ai-lab', os.getcwd()]:
    if p not in sys.path and os.path.isdir(os.path.join(p, 'src')):
        sys.path.insert(0, p)
        break

from src.config import (DATASET_PATH, RESULTS_DIR, REPORT_PATH, DEFAULT_MODEL_ID,
                        BASELINE_MODEL_ID, HF_TOKEN, MAX_NEW_TOKENS)
print('Repo on path. Config loaded.')
print('Dataset:', DATASET_PATH)
print('Results dir:', RESULTS_DIR)
print('Default model:', DEFAULT_MODEL_ID)
print('Baseline model:', BASELINE_MODEL_ID)
print('HF_TOKEN configured:', bool(HF_TOKEN))
print('Max new tokens:', MAX_NEW_TOKENS)


## 2 · Configuration and optional authentication

`HF_TOKEN` is only needed for gated Hugging Face models. In Colab, add it via **Secrets** (left sidebar → Secrets) and access it with `os.getenv("HF_TOKEN")`. Never paste a real token into a cell.

In [ ]:

from src.config import HF_TOKEN, DEFAULT_MODEL_ID
print('HF_TOKEN present:', bool(HF_TOKEN))
print('Model ID (override with CAREGIVER_MODEL_ID env var):', DEFAULT_MODEL_ID)


## 3 · Dataset creation/loading

Builds ~150 labeled synthetic examples (Arabic / Arabic+English / English) and validates them: duplicate IDs, missing fields, malformed rows. All data is fictional — for engineering practice only.

In [ ]:

from src.dataset import generate_dataset, write_dataset, load_dataset, validate_dataset, DATASET_PATH

examples = generate_dataset(150)
write_dataset(examples, DATASET_PATH)
report = validate_dataset(examples)
print(f'Examples: {report["count"]}')
print(f'Valid: {report["valid"]}')
print(f'Unique IDs: {report["unique_ids"]}')
print(f'Errors: {report["errors"][:3]}')
print()
print('Sample (EG-SYN-0001):')
import json
ex = examples[0]
for k in ['language_type', 'input_note', 'expected_facts', 'uncertainties', 'known_contradictions']:
    print(f'  {k}: {ex[k]!r}')


## 4 · Baseline implementation

Rule-based extractive baseline: care-domain keyword scoring over sentences. Runs offline with no API key — the reference point for judging the generative model.

In [ ]:

from src.models.baseline import RuleBasedBaseline
import json

baseline = RuleBasedBaseline()
print('Baseline available:', baseline.available())
note = examples[0]['input_note']
res = baseline.predict(note)
print('Input:', note)
print('Summary:', res.summary)
print(json.dumps(res.structured, ensure_ascii=False, indent=2))
print('Latency:', res.latency_s, 's | OK:', res.ok)


## 5 · Generative-model setup

Loads a configurable Hugging Face seq2seq model (`CAREGIVER_MODEL_ID`, default AraT5v2-base-102M). If loading fails (no network, gated model, OOM), the baseline still works and the error is reported.

In [ ]:

from src.models.huggingface_model import HfGenerativeModel

hf = HfGenerativeModel()
print('Model:', hf.name())
print('Available:', hf.available())
if not hf.available():
    print('Load error:', hf._load_error)


## 6 · Single-example inference

Run both available models on one fictional note and compare structured outputs.

In [ ]:

from src.models.huggingface_model import HfGenerativeModel
import json

note = "أم خالد أكلت كمية قليلة ولم تنم جيداً، ومشت بمفردها داخل البيت."
print('INPUT:', note)
print()

res_b = baseline.predict(note)
print('=== BASELINE ===')
print(res_b.summary)
print(json.dumps(res_b.structured, ensure_ascii=False, indent=2))

hf = HfGenerativeModel()
if hf.available():
    res_h = hf.predict(note)
    print('=== HF GENERATIVE ===')
    print(res_h.summary)
    print(json.dumps(res_h.structured, ensure_ascii=False, indent=2))
    print('Valid JSON:', res_h.valid_json, '| Error:', res_h.error)
else:
    print('HF model unavailable — baseline result shown above.')


## 7 · Evaluation on a fixed test set

Runs every available model over the first N examples and computes real metrics: JSON validity, schema compliance, fact recall (heuristic), unsupported-claim rate (heuristic), uncertainty/contradiction preservation, latency, failure rate.

In [ ]:

from src.evaluation.metrics import evaluate_all
from src.models.baseline import RuleBasedBaseline
from src.models.huggingface_model import HfGenerativeModel



In [ ]:

from src.dataset import load_dataset
from src.evaluation.metrics import evaluate_all
from src.models.baseline import RuleBasedBaseline
from src.models.huggingface_model import HfGenerativeModel

examples = load_dataset(DATASET_PATH)
limit = 30
models = [RuleBasedBaseline()] + ([HfGenerativeModel()] if HfGenerativeModel().available() else [])

results = []
for model in models:
    for ex in examples[:limit]:
        r = model.predict(ex['input_note'])
        rec = r.to_record()
        rec['example_id'] = ex['example_id']
        rec['language_type'] = ex['language_type']
        results.append(rec)

metrics = evaluate_all(results, {ex['example_id']: ex for ex in examples})

print(f'Evaluated {len(results)} runs across {[m.name() for m in models]}')
print(f'Valid JSON rate:       {metrics["json_valid_rate"]}')
print(f'Schema compliance:     {metrics["schema_compliance_rate"]}')
print(f'Failure rate:          {metrics["failure_rate"]["rate"]} ({metrics["failure_rate"]["failures"]}/{metrics["failure_rate"]["count"]})')
print(f'Mean latency:          {metrics["latency"]["mean_s"]} s')
print(f'Fact recall (heuristic): {metrics["fact_recall"]["mean_recall"]}')
print(f'Uncertainty preservation: {metrics["uncertainty_preservation"]["rate"]}')
print(f'Contradiction preservation: {metrics["contradiction_preservation"]["rate"]}')
print(f'Unsupported-claim rate (heuristic): {metrics["unsupported_claims"]["rate"]}')


## 8 · Metric calculations — closer look

Inspect per-metric detail: which examples omitted facts, which claims lack evidence.

In [ ]:

print('Fact recall detail:')
print('  omitted-fact examples:', metrics['fact_recall']['omitted_fact_examples'][:10])
print()
print('Unsupported claims detail:')
uc = metrics['unsupported_claims']
print('  unsupported/total claims:', uc['unsupported_claims'], '/', uc['total_claims'])
print('  flagged examples:', uc['flagged_example_ids'][:10])
print()
print('Uncertainty preservation:', metrics['uncertainty_preservation'])
print('Contradiction preservation:', metrics['contradiction_preservation'])
print('Latency stats:', metrics['latency'])


## 9 · Error analysis

Find a weak example and compare the reference annotation with the model output.

In [ ]:

import json
# Pick an example where baseline recall < 1
worst = None
for ex in examples[:limit]:
    r = baseline.predict(ex['input_note'])
    # crude: share of expected facts recalled
    facts = ex['expected_facts']
    if not facts:
        continue
    norm_out = (r.summary + ' ' + ' '.join(o.get('statement','') for o in r.structured.get('observations',[]))).lower()
    hits = sum(1 for f in facts if all(w in norm_out for w in f.split() if len(w) > 2)) / len(facts)
    if worst is None or hits < worst[0]:
        worst = (hits, ex, r)

hits, ex, r = worst
print('Worst recall example:', ex['example_id'], '| recall:', round(hits, 2))
print()
print('INPUT NOTE:')
print(' ', ex['input_note'])
print()
print('REFERENCE:')
print(json.dumps({k: ex[k] for k in ['expected_facts','important_details','uncertainties','known_contradictions','expected_summary','expected_evidence_quotes']}, ensure_ascii=False, indent=2))
print()
print('BASELINE OUTPUT:')
print(r.summary)
print(json.dumps(r.structured, ensure_ascii=False, indent=2))


## 10 · Model comparison

Same notes through baseline and (if available) the generative model, side by side.

In [ ]:

compare_notes = examples[:3]
for ex in compare_notes:
    print('='*70)
    print('ID:', ex['example_id'], '| Lang:', ex['language_type'])
    print('NOTE:', ex['input_note'])
    rb = baseline.predict(ex['input_note'])
    print('\nBASELINE:', rb.summary)
    if hf.available():
        rh = hf.predict(ex['input_note'])
        print('HF:', rh.summary, '| valid JSON:', rh.valid_json)
    print()


## 11 · Save results and generate report

Writes `results/experiment_results.jsonl`, `results/human_review_export.csv`, and `results/experiment_report.md`.

In [ ]:

from src.export import export_results_jsonl, export_review_csv
from src.evaluation.report import generate_report, save_report

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

export_results_jsonl(results, RESULTS_DIR / 'experiment_results.jsonl')
export_review_csv(results, RESULTS_DIR / 'human_review_export.csv')

md_report = generate_report(
    results=results,
    metrics=metrics,
    dataset_version='v1.0-synthetic',
    model_ids=[m.name() for m in models],
    config_snapshot={'max_new_tokens': MAX_NEW_TOKENS, 'max_input_chars': 2000},
)
save_report(md_report, REPORT_PATH)

print('Written:')
for f in sorted(RESULTS_DIR.iterdir()):
    print(' ', f.name, f.stat().st_size, 'bytes')
print()
print(md_report)


## 12 · Optional: launch the Gradio interface

Uncomment to run the UI inside Colab (gives a public share URL). Otherwise run `python app.py` locally.

In [ ]:

# from app import build_app
# demo = build_app()
# demo.launch(server_name="0.0.0.0", server_port=7860, share=True)
print('To launch the UI locally:')
print('  cd caregiver-ai-lab && python app.py')
print('Then open http://localhost:7860')
print('Tabs: Try a Note · Compare Models · Evaluation Dashboard · Error Analysis · Learn')
